In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install xgboost lightgbm catboost --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.5 MB/s eta 0:00:00


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("catboost not installed - run '!pip install catboost --quiet' first.")

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("xgboost not installed - run '!pip install xgboost --quiet' first.")

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False
    print("lightgbm not installed - run '!pip install lightgbm --quiet' first.")


# PATHS

FEATURES_DIR = "/content/drive/MyDrive/ADHD_EEG_Project/data/features_paper_method"
RESULTS_DIR = "/content/drive/MyDrive/ADHD_EEG_Project/results_rfe_smote_final"
os.makedirs(RESULTS_DIR, exist_ok=True)


FC_MEASURES = ["Correlation", "Coherence", "PLV", "PLI"]

LABELS_PATH = os.path.join(FEATURES_DIR, "labels_paper.npy")

RFE_N_FEATURES = 20   # fixed, no search (half of 40 features)
RFE_STEP = 5           # avoids slow step=1 default
SMOTE_K_NEIGHBORS = 5

CV = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# FIXED CLASSIFIER CONFIGS

CLASSIFIER_DEFS = {
    "LogisticRegression": lambda: LogisticRegression(
        penalty="l2", C=1.0, solver="liblinear", class_weight="balanced",
        max_iter=5000, random_state=42
    ),
    "SVM": lambda: SVC(
        kernel="linear", C=1.0, class_weight="balanced", random_state=42
    ),
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=200, max_depth=None, class_weight="balanced", random_state=42
    ),
}

if HAS_XGBOOST:
    CLASSIFIER_DEFS["XGBoost"] = lambda: XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=42
    )

if HAS_LIGHTGBM:
    CLASSIFIER_DEFS["LightGBM"] = lambda: LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        class_weight="balanced", random_state=42, verbose=-1
    )

if HAS_CATBOOST:
    CLASSIFIER_DEFS["CatBoost"] = lambda: CatBoostClassifier(
        iterations=200, depth=6, learning_rate=0.05,
        auto_class_weights="Balanced", random_state=42, verbose=0
    )

# PIPELINE BUILDER: imputer -> SMOTE (train folds only) -> scaler -> RFE -> clf

def build_pipeline(clf_factory):
    steps = [
        ("imputer", SimpleImputer(strategy="mean")),
        ("smote", SMOTE(random_state=42, k_neighbors=SMOTE_K_NEIGHBORS)),
        ("scaler", StandardScaler()),
        ("reduce", RFE(
            estimator=clf_factory(),
            n_features_to_select=RFE_N_FEATURES,
            step=RFE_STEP
        )),
        ("clf", clf_factory()),
    ]
    return ImbPipeline(steps)

# EVALUATION FUNCTION

def evaluate_classifier(X, y, clf_factory):
    pipeline_template = build_pipeline(clf_factory)

    fold_accuracy, fold_precision, fold_recall, fold_f1, fold_auc = [], [], [], [], []
    total_tp = total_tn = total_fp = total_fn = 0

    for train_idx, test_idx in CV.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = clone(pipeline_template)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        if hasattr(model, "decision_function"):
            y_scores = model.decision_function(X_test)
        else:
            y_scores = model.predict_proba(X_test)[:, 1]

        fold_accuracy.append(accuracy_score(y_test, y_pred))
        fold_precision.append(precision_score(y_test, y_pred, zero_division=0))
        fold_recall.append(recall_score(y_test, y_pred, zero_division=0))
        fold_f1.append(f1_score(y_test, y_pred, zero_division=0))
        fold_auc.append(roc_auc_score(y_test, y_scores))

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
        total_tp += tp; total_tn += tn; total_fp += fp; total_fn += fn

    return {
        "Accuracy Mean": np.mean(fold_accuracy), "Accuracy SD": np.std(fold_accuracy),
        "Precision Mean": np.mean(fold_precision), "Precision SD": np.std(fold_precision),
        "Recall Mean": np.mean(fold_recall), "Recall SD": np.std(fold_recall),
        "F1 Mean": np.mean(fold_f1), "F1 SD": np.std(fold_f1),
        "AUC Mean": np.mean(fold_auc), "AUC SD": np.std(fold_auc),
        "TP": total_tp, "TN": total_tn, "FP": total_fp, "FN": total_fn,
    }

# MAIN LOOP: for each FC measure, run all 6 classifiers

y_labels = np.load(LABELS_PATH)
print(f"Loaded labels: shape {y_labels.shape}, "
      f"Total Subjects: {len(y_labels)}, Control={np.sum(y_labels==0)}, ADHD={np.sum(y_labels==1)}")
print(f"Classifiers to run: {list(CLASSIFIER_DEFS.keys())}")
print(f"FC measures to run: {FC_MEASURES}")
print(f"Fixed settings: RFE n_features_to_select={RFE_N_FEATURES}, step={RFE_STEP}, "
      f"SMOTE k_neighbors={SMOTE_K_NEIGHBORS}, 10-fold CV (no repeats), no GridSearchCV\n")

all_rows = []

for measure in FC_MEASURES:
    feature_path = os.path.join(FEATURES_DIR, f"{measure}_features_paper.npy")
    if not os.path.exists(feature_path):
        print(f"[SKIPPED] {measure}: file not found at {feature_path}")
        continue

    X = np.load(feature_path)
    y = y_labels
    print(f"\n=== FC measure: {measure} | shape {X.shape}, "
          f"NaN values: {int(np.isnan(X).sum())} ===")

    measure_rows = []
    for clf_name, clf_factory in CLASSIFIER_DEFS.items():
        start_time = time.time()
        metrics = evaluate_classifier(X, y, clf_factory)
        elapsed = time.time() - start_time

        print(f"  {clf_name:20s} | "
              f"Acc={metrics['Accuracy Mean']:.4f}+/-{metrics['Accuracy SD']:.4f} | "
              f"AUC={metrics['AUC Mean']:.4f}+/-{metrics['AUC SD']:.4f} | {elapsed:.1f}s")

        row = {"FC Measure": measure, "Classifier": clf_name, **metrics}
        measure_rows.append(row)
        all_rows.append(row)

    # Save per-measure results as we go (also acts as a checkpoint)
    measure_df = pd.DataFrame(measure_rows).sort_values("AUC Mean", ascending=False).reset_index(drop=True)
    measure_csv_path = os.path.join(RESULTS_DIR, f"results_{measure}.csv")
    measure_df.to_csv(measure_csv_path, index=False)

    # Checkpoint the combined table after every measure too
    pd.DataFrame(all_rows).to_csv(
        os.path.join(RESULTS_DIR, "results_ALL_MEASURES_PARTIAL.csv"), index=False
    )

# FINAL COMBINED TABLE


final_df = pd.DataFrame(all_rows)
final_df = final_df.sort_values(["FC Measure", "AUC Mean"], ascending=[True, False]).reset_index(drop=True)

print("\n\n" + "=" * 110)
print("FINAL RESULTS - RFE + SMOTE only, all 6 classifiers x all FC measures, plain 10-fold CV")
print("=" * 110)
print(final_df.round(4).to_string(index=False))
print("=" * 110)

# Overall best row (for quick reference / highlighting in the paper)
best_row = final_df.sort_values("AUC Mean", ascending=False).iloc[[0]]
print("\nBest single result overall:")
print(best_row.round(4).to_string(index=False))

final_csv_path = os.path.join(RESULTS_DIR, "results_ALL_MEASURES_FINAL.csv")
final_df.to_csv(final_csv_path, index=False)
print(f"\nSaved final combined table to: {final_csv_path}")
print("Per-measure tables saved individually as results_<Measure>.csv in the same folder.")

Loaded labels: shape (79,), Total Subjects: 79, Control=42, ADHD=37
Classifiers to run: ['LogisticRegression', 'SVM', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']
FC measures to run: ['Correlation', 'Coherence', 'PLV', 'PLI']
Fixed settings: RFE n_features_to_select=20, step=5, SMOTE k_neighbors=5, 10-fold CV (no repeats), no GridSearchCV


=== FC measure: Correlation | shape (79, 40), NaN values: 0 ===
  LogisticRegression   | Acc=0.5679+/-0.1648 | AUC=0.5925+/-0.1899 | 0.3s
  SVM                  | Acc=0.5554+/-0.1549 | AUC=0.5487+/-0.1802 | 0.3s
  RandomForest         | Acc=0.6339+/-0.1395 | AUC=0.6960+/-0.1573 | 20.7s
  XGBoost              | Acc=0.6232+/-0.1651 | AUC=0.6769+/-0.2049 | 7.8s
  LightGBM             | Acc=0.4179+/-0.1119 | AUC=0.4435+/-0.1807 | 1.4s
  CatBoost             | Acc=0.6589+/-0.1570 | AUC=0.7548+/-0.1673 | 34.1s

=== FC measure: Coherence | shape (79, 40), NaN values: 0 ===
  LogisticRegression   | Acc=0.5929+/-0.1803 | AUC=0.6471+/-0.1696 | 0.4s
  SV